Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools

Calling the Libraries:

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein/vein001_1/01.jpg'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Train

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# Define base paths
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
stride = 10
NUM_SUBJECTS = 123
NUM_FINGERS = 4
NUM_IMAGES = 4
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
NUM_PCS = None  # Use full PCA space unless truncating

# === RIU2 MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        if transitions <= 2:
            table[i] = sum(min_rotation)
        else:
            table[i] = P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP HISTOGRAM ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === TRAINING ===
train_lbp_features = []
train_labels = []

print("\n🔄 Extracting RIU2-LBP features from FV-USM (P1-S1, session-labeled)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Subjects"):
    for img_idx in range(1, NUM_IMAGES + 1):
        for session_path in [base_path_sess1, base_path_sess2]:
            fused_vector = []
            complete = True
            session_name = "1st" if "1st" in session_path else "2nd"

            print(f"\n--- Extracting {session_name.upper()} SESSION for subject {subject_id:03d}, image {img_idx:02d} ---")

            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(session_path, folder_name, f"{img_idx:02d}.jpg")

                print(f"✅ Using image: {img_path}")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Missing: {img_path}")
                    complete = False
                    break

                img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img).astype(np.float64) / 255.0
                img = (img - np.mean(img)) / (np.std(img) + 1e-8)

                for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                    for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        block_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            block_hist.extend(hist)
                        fused_vector.extend(block_hist)

            if complete and len(fused_vector) > 0:
                label = f"{subject_id:03d}_img{img_idx:02d}_fused_{session_name}"
                train_lbp_features.append(fused_vector)
                train_labels.append(label)

# === NORMALIZATION & PCA ===
train_lbp_features = np.array(train_lbp_features, dtype=np.float32)
train_lbp_features = normalize(train_lbp_features, norm='l2')
train_labels = np.array(train_labels)

print("\n✅ Fused feature matrix shape:", train_lbp_features.shape)

# === PCA (Gram Matrix Based) ===
mean_vector = np.mean(train_lbp_features, axis=0)
centered_data = train_lbp_features - mean_vector
gram_matrix = centered_data @ centered_data.T

eig_vals, eig_vecs = np.linalg.eigh(gram_matrix)
sorted_indices = np.argsort(-eig_vals)
eig_vals = eig_vals[sorted_indices]
eig_vecs = eig_vecs[:, sorted_indices]

valid = eig_vals > 1e-10
eig_vals_valid = eig_vals[valid]
eig_vecs_valid = eig_vecs[:, valid]

eig_vecs_full = (centered_data.T @ eig_vecs_valid) / np.sqrt(eig_vals_valid)

if NUM_PCS is not None and NUM_PCS < eig_vecs_full.shape[1]:
    eig_vecs_full = eig_vecs_full[:, :NUM_PCS]

train_lbp_pca = centered_data @ eig_vecs_full

print("✅ PCA-transformed feature shape:", train_lbp_pca.shape)
print("✅ Number of principal components:", eig_vecs_full.shape[1])


Test

In [ ]:
# === TEST SET EXTRACTION ===
test_lbp_features = []
test_labels = []

NUM_TEST_IMAGES = [5, 6]

print("\n🔄 Extracting TEST features from FV-USM (Session-labeled, P1-S1)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Test Subjects"):
    for img_idx in NUM_TEST_IMAGES:
        for session_path in [base_path_sess1, base_path_sess2]:
            session_name = "1st" if "1st" in session_path else "2nd"
            fused_vector = []
            complete = True

            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(session_path, folder_name, f"{img_idx:02d}.jpg")

                print(f"🔍 Trying image: {img_path}")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Missing: {img_path}")
                    complete = False
                    break

                img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img).astype(np.float64) / 255.0
                img = (img - np.mean(img)) / (np.std(img) + 1e-8)

                for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                    for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        block_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            block_hist.extend(hist)
                        fused_vector.extend(block_hist)

            if complete and len(fused_vector) > 0:
                label = f"{subject_id:03d}_img{img_idx:02d}_fused_{session_name}"
                test_lbp_features.append(fused_vector)
                test_labels.append(label)

# === Normalize and project ===
test_lbp_features = np.array(test_lbp_features, dtype=np.float32)
test_lbp_features = normalize(test_lbp_features, norm='l2')
test_labels = np.array(test_labels)

centered_test_data = test_lbp_features - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full

print("\n✅ Test feature shape (pre-PCA):", test_lbp_features.shape)
print("✅ Test feature shape (post-PCA):", proj_test_data.shape)
print("✅ Total test samples collected:", len(test_labels))


Benchmarking 1

In [ ]:
# === SESSION-SENSITIVE EVALUATION ===
correct_matches = 0
total_tests = len(test_labels)

print("\n🔍 Starting Session-Sensitive Classification using Manhattan distance...")

for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]  # e.g., "123_img06_fused_2nd"

    # 📏 Manhattan distance to all training vectors
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)

    # 🏆 Closest training sample
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "123_img04_fused_1st"

    # 🎯 Check session-sensitive match: subject ID and session must match
    true_parts = true_label.split('_')       # ['123', 'img06', 'fused', '2nd']
    pred_parts = predicted_label.split('_')  # ['123', 'img04', 'fused', '1st']

    true_id, true_session = true_parts[0], true_parts[-1]
    pred_id, pred_session = pred_parts[0], pred_parts[-1]

    if (true_id == pred_id) and (true_session == pred_session):
        correct_matches += 1
        result = "✅ CORRECT"
    else:
        result = "❌ WRONG"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {result}")

# 📈 Final Accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎯 Session-Sensitive Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")


Benchmarking 2

In [ ]:
correct_matches = 0
total_tests = len(test_labels)

# 📉 Project test features into PCA space
centered_test_data = test_lbp_features - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full

print("\n🔍 Starting Session-Independent Classification using Manhattan distance...")

for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]  # e.g., "007_img06_fused_2nd"

    # 📏 Manhattan distance to all training samples
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)

    # 🏆 Find closest match
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "007_img04_fused_1st"

    # 🎯 Extract subject IDs only
    true_id = true_label.split('_')[0]
    pred_id = predicted_label.split('_')[0]

    if pred_id == true_id:
        correct_matches += 1
        match_result = "✅"
    else:
        match_result = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# 📈 Final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Session-Independent Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")


Session Sensitive R5

In [ ]:
import numpy as np
from collections import defaultdict

ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC (Rank-1 & Rank-5)...")

for i in range(total_tests):
    test_label = test_labels[i]
    proj_test = proj_test_data[i]

    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_session}"

    # 📏 Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = train_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_session = parts[-1]
        candidate_id = f"{candidate_subject}_{candidate_session}"

        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# === FINAL RESULTS ===
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject + Session): {accuracy:.2f}%")


Session Sensitive CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC Curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = proj_test_data[i]
    test_parts = test_labels[i].split('_')
    true_subject = test_parts[0]
    true_session = test_parts[-1]
    true_id = f"{true_subject}_{true_session}"

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        cand_parts = candidate_label.split('_')
        cand_subject = cand_parts[0]
        cand_session = cand_parts[-1]
        candidate_id = f"{cand_subject}_{cand_session}"

        if candidate_id == true_id:
            rank_correct[r:] += 1
            break

# === Normalize
cmc_curve = (rank_correct / total_tests) * 100

# === Plot
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Sensitive CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Sensitive CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) + PCA (Strategy 1, Protocol 1)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank+1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Print key ranks
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Sensitive Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter

# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMOTE


all_scores = []
all_labels = []

# === Loop through test and train samples
for i in range(len(proj_test_data)):
    test_vec = proj_test_data[i]
    test_parts = test_labels[i].split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]

    for j in range(len(train_lbp_pca)):
        train_vec = train_lbp_pca[j]
        train_parts = train_labels[j].split('_')
        train_subject = train_parts[0]
        train_session = train_parts[-1]

        # === Compute negative Manhattan distance (higher = more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # === Label as genuine if both subject and session match
        is_genuine = int(test_subject == train_subject and test_session == train_session)
        all_labels.append(is_genuine)

# === Convert to NumPy arrays
scores = np.array(all_scores).reshape(-1, 1)
labels = np.array(all_labels)

print("🔢 Original label distribution:", Counter(labels))

# === Normalize scores to [0, 1]
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Apply SMOTE (optional)
if use_smote:
    smote = SMOTE(random_state=42)
    scores, labels = smote.fit_resample(scores, labels)
    print("🧪 After SMOTE label distribution:", Counter(labels))

# === Threshold Sweeping to find best F1
best_f1 = best_thresh = best_prec = best_rec = 0
for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = prec
        best_rec = rec

# === Final evaluation
final_preds = (scores >= best_thresh).astype(int)
acc = accuracy_score(labels, final_preds)

# === Print results
print("\n🔍 Summary (Session-Sensitive)")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {acc * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")


Session Independent R5

In [ ]:
from collections import defaultdict
import numpy as np

ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating CMC, Rank-1, and Rank-5 (Session-Independent)...")

for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_id = test_labels[i].split('_')[0]  # 🔍 Subject only

    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)
    top_k_indices = np.argsort(distances)

    found = False
    for r in range(1, max(ranks) + 1):
        candidate_label = train_labels[top_k_indices[r - 1]]
        candidate_id = candidate_label.split('_')[0]  # 🔍 Subject only

        if candidate_id == true_id and not found:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            found = True

# === Final Results ===
for k in ranks:
    cmc_score = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy: {cmc_score:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
correct_matches = 0
total_tests = len(test_labels)

print("📊 Calculating Session-Independent CMC Curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_subject = test_labels[i].split('_')[0]  # Subject ID only

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # === Rank-1 Classification (with subject matching)
    for idx in sorted_indices:
        pred_subject = train_labels[idx].split('_')[0]
        if pred_subject == true_subject:
            if idx == sorted_indices[0]:
                correct_matches += 1  # ✅ Only if correct match is at Rank-1
            break  # Stop after first match for classification

    # === CMC Curve (match subject at correct rank)
    for r in range(max_rank):
        candidate_subject = train_labels[sorted_indices[r]].split('_')[0]
        if candidate_subject == true_subject:
            rank_correct[r:] += 1
            break

# === Normalize CMC to percentage
cmc_curve = (rank_correct / total_tests) * 100
rank1_accuracy = (correct_matches / total_tests) * 100

# === Plotting
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Independent CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) + PCA (Strategy 1, Protocol 1)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Final Report
print("\n📊 Session-Independent Classification Report")
print(f"✅ Rank-1 Classification Accuracy : {rank1_accuracy:.2f}%")
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter

# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMOTE



all_scores = []
all_labels = []

# === Loop through test and train samples
for i in range(len(proj_test_data)):
    test_vec = proj_test_data[i]
    test_parts = test_labels[i].split('_')
    test_subject = test_parts[0]  # Only subject ID

    for j in range(len(train_lbp_pca)):
        train_vec = train_lbp_pca[j]
        train_parts = train_labels[j].split('_')
        train_subject = train_parts[0]  # Only subject ID

        # === Compute negative Manhattan distance (higher = more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # === Label as genuine if subject IDs match (ignore session)
        is_genuine = int(test_subject == train_subject)
        all_labels.append(is_genuine)

# === Convert to NumPy arrays
scores = np.array(all_scores).reshape(-1, 1)
labels = np.array(all_labels)

print("🔢 Original label distribution:", Counter(labels))

# === Normalize scores to [0, 1]
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Apply SMOTE (optional)
if use_smote:
    smote = SMOTE(random_state=42)
    scores, labels = smote.fit_resample(scores, labels)
    print("🧪 After SMOTE label distribution:", Counter(labels))

# === Threshold Sweeping to find best F1
best_f1 = best_thresh = best_prec = best_rec = 0
for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = prec
        best_rec = rec

# === Final evaluation
final_preds = (scores >= best_thresh).astype(int)
acc = accuracy_score(labels, final_preds)

# === Print results
print("\n🔍 Summary (Session-Independent)")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {acc * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
